# 06 SHAP-LIME Explanation Disagreement – Configurable Pipeline

## Purpose

This notebook compares SHAP and LIME explanations for the same models, same test instances, and same prediction task.

It is designed as a configurable pipeline for:

- German Credit
- Pima Diabetes
- COMPAS

Only the `ACTIVE_DATASET` value should be changed.

## Why this notebook matters

SHAP and LIME are two independent explanation methods. This notebook asks:

> Do SHAP and LIME identify similar important features for the same prediction?

The output becomes evidence of **cross-method explanation consistency** and is now used as part of the EES Fidelity pillar.

## Link to EES Fidelity

The revised Fidelity pillar uses three equal-weighted evidence sources:

```text
Fidelity Score =
1/3 SHAP Reconstruction Fidelity
+ 1/3 LIME Local Surrogate Fidelity
+ 1/3 SHAP-LIME Agreement Score
```

This notebook provides the third component:

```text
SHAP-LIME Agreement Score
```

Kendall's Tau ranges from -1 to +1, so it is converted to a 0–1 score:

```text
agreement_score = (kendall_tau + 1) / 2
```

In [ ]:
# =========================================================
# 0. Dataset Configuration
# =========================================================

# Select only one active dataset.
# ACTIVE_DATASET = "german_credit"
# ACTIVE_DATASET = "pima_diabetes"
ACTIVE_DATASET = "compas"

DATASET_CONFIGS = {
    "german_credit": {
        "dataset_name": "german_credit",
        "dataset_label": "German Credit",
        "target_name": "target",
        "positive_class_label": "Default",
        "negative_class_label": "Non-default",
        "results_folder": "german_credit",
        "shap_folder": "04_shap_interpretation_german_credit",
        "lime_folder": "05_lime_interpretation_german_credit",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_german_credit"
    },

    "pima_diabetes": {
        "dataset_name": "pima_diabetes",
        "dataset_label": "Pima Diabetes",
        "target_name": "target",
        "positive_class_label": "Diabetes",
        "negative_class_label": "No diabetes",
        "results_folder": "pima_diabetes",
        "shap_folder": "04_shap_interpretation_pima_diabetes",
        "lime_folder": "05_lime_interpretation_pima_diabetes",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_pima_diabetes"
    },

    "compas": {
        "dataset_name": "compas",
        "dataset_label": "COMPAS",
        "target_name": "target",
        "positive_class_label": "Recidivism",
        "negative_class_label": "No recidivism",
        "results_folder": "compas",
        "shap_folder": "04_shap_interpretation_compas",
        "lime_folder": "05_lime_interpretation_compas",
        "disagreement_folder": "06_shap_lime_explaination_disagreement_compas"
    }
}

CONFIG = DATASET_CONFIGS[ACTIVE_DATASET]

print("Active dataset:", CONFIG["dataset_label"])
print("Dataset name:", CONFIG["dataset_name"])

Active dataset: COMPAS
Dataset name: compas


In [ ]:
# =========================================================
# 1. Import Required Libraries
# =========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
from datetime import datetime
import json
import numpy as np
import pandas as pd

from scipy.stats import kendalltau

pd.set_option("display.max_columns", 250)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
rng = np.random.RandomState(RANDOM_STATE)

print("Libraries imported successfully.")
print("Random seed:", RANDOM_STATE)

Libraries imported successfully.
Random seed: 42


In [ ]:
# =========================================================
# 2. Mount Google Drive and Configure Paths
# =========================================================

from google.colab import drive

try:
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped or already available:", e)

ROOT_PATH = Path("/content/drive/MyDrive/Master_Research")

DATASET_NAME = CONFIG["dataset_name"]
NOTEBOOK_NAME = CONFIG["disagreement_folder"]

RESULTS_ROOT = ROOT_PATH / "07_Results" / CONFIG["results_folder"]

SHAP_RESULTS_PATH = RESULTS_ROOT / CONFIG["shap_folder"]
SHAP_TABLES_PATH = SHAP_RESULTS_PATH / "tables"
SHAP_DATASETS_PATH = SHAP_RESULTS_PATH / "datasets"

LIME_RESULTS_PATH = RESULTS_ROOT / CONFIG["lime_folder"]
LIME_TABLES_PATH = LIME_RESULTS_PATH / "tables"
LIME_DATASETS_PATH = LIME_RESULTS_PATH / "datasets"

RESULTS_PATH = RESULTS_ROOT / NOTEBOOK_NAME
TABLES_PATH = RESULTS_PATH / "tables"
DATASETS_PATH = RESULTS_PATH / "datasets"
LOGS_PATH = RESULTS_PATH / "logs"

for path in [RESULTS_PATH, TABLES_PATH, DATASETS_PATH, LOGS_PATH]:
    path.mkdir(parents=True, exist_ok=True)

print("Results root:", RESULTS_ROOT)
print("SHAP path:", SHAP_RESULTS_PATH)
print("LIME path:", LIME_RESULTS_PATH)
print("Output path:", RESULTS_PATH)

Mounted at /content/drive
Results root: /content/drive/MyDrive/Master_Research/07_Results/compas
SHAP path: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas
LIME path: /content/drive/MyDrive/Master_Research/07_Results/compas/05_lime_interpretation_compas
Output path: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas


In [ ]:
# =========================================================
# 3. Helper Functions
# =========================================================

def read_csv_safely(path, label, required=True):
    path = Path(path)
    if not path.exists():
        if required:
            raise FileNotFoundError(f"{label} not found at: {path}")
        print(f"Optional file not found: {label} at {path}")
        return pd.DataFrame()
    return pd.read_csv(path)


def find_first_existing_file(candidate_paths, label, required=True):
    for path in candidate_paths:
        path = Path(path)
        if path.exists():
            print(f"{label} found at: {path}")
            return path
    if required:
        checked = "\n".join(str(path) for path in candidate_paths)
        raise FileNotFoundError(f"{label} not found. Checked:\n{checked}")
    print(f"{label} not found.")
    return None


def save_df(df, filename, folder=None, index=False):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    df.to_csv(output_path, index=index)
    print(f"Saved: {output_path}")
    return output_path


def save_excel(dfs, filename, folder=None):
    if folder is None:
        folder = TABLES_PATH
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    output_path = folder / filename
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        for sheet_name, df in dfs.items():
            if isinstance(df, pd.DataFrame):
                df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
    print(f"Saved: {output_path}")
    return output_path


def normalise_kendall_tau(tau):
    if pd.isna(tau):
        return np.nan
    return (tau + 1) / 2


def classify_agreement(score):
    if pd.isna(score):
        return "Not available"
    if score >= 0.85:
        return "Very strong agreement"
    if score >= 0.70:
        return "Strong agreement"
    if score >= 0.55:
        return "Moderate agreement"
    if score >= 0.40:
        return "Weak agreement"
    return "Poor agreement"


def standardise_lime_feature_name(feature_rule, shap_features):
    feature_rule = str(feature_rule)
    matching_features = [
        feature for feature in shap_features
        if feature_rule == feature
        or feature_rule.startswith(feature + " ")
        or (" " + feature + " ") in feature_rule
        or feature in feature_rule
    ]
    if len(matching_features) == 0:
        return feature_rule
    return sorted(matching_features, key=len, reverse=True)[0]


def top_k_rank_table(df, value_col, feature_col, k):
    temp = (
        df
        .sort_values(value_col, ascending=False)
        .head(k)
        .copy()
        .reset_index(drop=True)
    )
    temp["rank"] = np.arange(1, len(temp) + 1)
    return temp[[feature_col, value_col, "rank"]]


def calculate_rank_agreement(shap_instance_df, lime_instance_df, k):
    shap_top = top_k_rank_table(
        shap_instance_df,
        value_col="abs_shap_value",
        feature_col="feature",
        k=k
    ).rename(columns={"rank": "shap_rank"})

    lime_top = top_k_rank_table(
        lime_instance_df,
        value_col="abs_lime_weight",
        feature_col="feature",
        k=k
    ).rename(columns={"rank": "lime_rank"})

    shap_features = set(shap_top["feature"])
    lime_features = set(lime_top["feature"])
    overlap_features = sorted(shap_features.intersection(lime_features))
    union_features = sorted(shap_features.union(lime_features))

    top_k_overlap_count = len(overlap_features)
    top_k_overlap_ratio = top_k_overlap_count / k if k > 0 else np.nan

    if len(union_features) < 2:
        kendall_tau = np.nan
        kendall_pvalue = np.nan
    else:
        rank_default = k + 1
        shap_rank_map = dict(zip(shap_top["feature"], shap_top["shap_rank"]))
        lime_rank_map = dict(zip(lime_top["feature"], lime_top["lime_rank"]))
        shap_rank_vector = [shap_rank_map.get(feature, rank_default) for feature in union_features]
        lime_rank_vector = [lime_rank_map.get(feature, rank_default) for feature in union_features]
        kendall_tau, kendall_pvalue = kendalltau(shap_rank_vector, lime_rank_vector)

    agreement_score = normalise_kendall_tau(kendall_tau)

    return {
        "kendall_tau": kendall_tau,
        "kendall_pvalue": kendall_pvalue,
        "agreement_score": agreement_score,
        "agreement_band": classify_agreement(agreement_score),
        "top_k_overlap_count": top_k_overlap_count,
        "top_k_overlap_ratio": top_k_overlap_ratio,
        "n_shap_top_features": len(shap_features),
        "n_lime_top_features": len(lime_features),
        "n_union_features": len(union_features),
        "n_overlap_features": len(overlap_features),
        "overlap_features": ", ".join(overlap_features)
    }

## 4. Load SHAP and LIME Outputs

This notebook expects:

- `all_test_local_shap_long.csv` from Notebook 04
- `lime_all_test_long.csv` from Notebook 05

Both should contain explanations for all test instances.

In [ ]:
# =========================================================
# 4. Load SHAP and LIME Outputs
# =========================================================

shap_file = find_first_existing_file(
    [
        SHAP_DATASETS_PATH / "all_test_local_shap_long.csv",
        SHAP_TABLES_PATH / "all_test_local_shap_long.csv",
        SHAP_DATASETS_PATH / "shap_all_test_long.csv",
        SHAP_TABLES_PATH / "shap_all_test_long.csv",
    ],
    "SHAP long output"
)

lime_file = find_first_existing_file(
    [
        LIME_DATASETS_PATH / "lime_all_test_long.csv",
        LIME_TABLES_PATH / "lime_all_test_long.csv",
    ],
    "LIME long output"
)

shap_long_df = read_csv_safely(shap_file, "SHAP long output")
lime_long_df = read_csv_safely(lime_file, "LIME long output")

print("SHAP long:", shap_long_df.shape)
print("LIME long:", lime_long_df.shape)

display(shap_long_df.head())
display(lime_long_df.head())

SHAP long output found at: /content/drive/MyDrive/Master_Research/07_Results/compas/04_shap_interpretation_compas/datasets/all_test_local_shap_long.csv
LIME long output found at: /content/drive/MyDrive/Master_Research/07_Results/compas/05_lime_interpretation_compas/datasets/lime_all_test_long.csv
SHAP long: (525000, 8)
LIME long: (175000, 7)


,model_id,model_name,test_instance_position,feature,shap_value,abs_shap_value,feature_value,explainer_type
0,logistic_regression,Logistic Regression,0,age,0.370608,0.370608,-0.437500,LinearExplainer
1,logistic_regression,Logistic Regression,0,juv_fel_count,-0.002708,0.002708,0.000000,LinearExplainer
2,logistic_regression,Logistic Regression,0,juv_misd_count,-0.007029,0.007029,0.000000,LinearExplainer
3,logistic_regression,Logistic Regression,0,juv_other_count,0.113646,0.113646,1.000000,LinearExplainer
4,logistic_regression,Logistic Regression,0,priors_count,-0.173139,0.173139,0.000000,LinearExplainer


,model_id,model_name,test_instance_position,feature_rule,feature,lime_weight,abs_lime_weight
0,logistic_regression,Logistic Regression,0,c_charge_degree_Missing <= 0.00,c_charge_degree_Missing <= 0.00,-0.383982,0.383982
1,logistic_regression,Logistic Regression,0,c_charge_degree_(CO3) <= 0.00,c_charge_degree_(CO3) <= 0.00,-0.314434,0.314434
2,logistic_regression,Logistic Regression,0,c_charge_degree_(F5) <= 0.00,c_charge_degree_(F5) <= 0.00,0.272423,0.272423
3,logistic_regression,Logistic Regression,0,c_charge_degree_(F6) <= 0.00,c_charge_degree_(F6) <= 0.00,0.201417,0.201417
4,logistic_regression,Logistic Regression,0,c_charge_degree_(NI0) <= 0.00,c_charge_degree_(NI0) <= 0.00,-0.199865,0.199865


## 5. Standardise LIME Feature Names

SHAP uses clean feature names, while LIME often returns feature rules.  
This section maps LIME feature rules back to the underlying feature names before rank comparison.

In [ ]:
# =========================================================
# 5. Standardise LIME Feature Names
# =========================================================

required_shap_cols = ["model_id", "model_name", "test_instance_position", "feature", "abs_shap_value"]
required_lime_cols = ["model_id", "model_name", "test_instance_position", "abs_lime_weight"]

if "feature" not in lime_long_df.columns and "feature_rule" in lime_long_df.columns:
    lime_long_df["feature"] = lime_long_df["feature_rule"]

required_lime_cols.append("feature")

missing_shap_cols = [col for col in required_shap_cols if col not in shap_long_df.columns]
missing_lime_cols = [col for col in required_lime_cols if col not in lime_long_df.columns]

if missing_shap_cols:
    raise ValueError(f"Missing SHAP columns: {missing_shap_cols}")

if missing_lime_cols:
    raise ValueError(f"Missing LIME columns: {missing_lime_cols}")

shap_features = sorted(shap_long_df["feature"].astype(str).unique().tolist())

lime_long_df = lime_long_df.copy()
lime_long_df["feature_rule_original"] = lime_long_df["feature"].astype(str)
lime_long_df["feature"] = lime_long_df["feature_rule_original"].apply(
    lambda x: standardise_lime_feature_name(x, shap_features)
)

feature_standardisation_audit_df = (
    lime_long_df[["feature_rule_original", "feature"]]
    .drop_duplicates()
    .sort_values(["feature", "feature_rule_original"])
    .reset_index(drop=True)
)

display(feature_standardisation_audit_df.head(30))
save_df(feature_standardisation_audit_df, "lime_feature_standardisation_audit.csv")

,feature_rule_original,feature
0,-0.38 < age <= 0.00,age
1,0.00 < age <= 0.62,age
2,age <= -0.38,age
3,age > 0.62,age
4,c_charge_degree_(CO3) <= 0.00,c_charge_degree_(CO3)
5,c_charge_degree_(CT) <= 0.00,c_charge_degree_(CT)
6,c_charge_degree_(F1) <= 0.00,c_charge_degree_(F1)
7,c_charge_degree_(F1) > 0.00,c_charge_degree_(F1)
8,c_charge_degree_(F2) <= 0.00,c_charge_degree_(F2)
9,c_charge_degree_(F2) > 0.00,c_charge_degree_(F2)


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/lime_feature_standardisation_audit.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/lime_feature_standardisation_audit.csv')

In [ ]:
# =========================================================
# 6. Input Validation
# =========================================================

validation_rows = []

validation_rows.append({"check": "SHAP output not empty", "passed": not shap_long_df.empty})
validation_rows.append({"check": "LIME output not empty", "passed": not lime_long_df.empty})
validation_rows.append({"check": "SHAP required columns available", "passed": len(missing_shap_cols) == 0})
validation_rows.append({"check": "LIME required columns available", "passed": len(missing_lime_cols) == 0})

shap_keys = set(zip(shap_long_df["model_id"], shap_long_df["test_instance_position"]))
lime_keys = set(zip(lime_long_df["model_id"], lime_long_df["test_instance_position"]))

validation_rows.append({
    "check": "At least one overlapping model-instance pair",
    "passed": len(shap_keys.intersection(lime_keys)) > 0
})

validation_rows.append({
    "check": "LIME mapped features are non-missing",
    "passed": lime_long_df["feature"].notna().all()
})

disagreement_input_validation_df = pd.DataFrame(validation_rows)

display(disagreement_input_validation_df)
save_df(disagreement_input_validation_df, "disagreement_input_validation.csv")

if not disagreement_input_validation_df["passed"].all():
    raise ValueError("One or more disagreement input validation checks failed.")

,check,passed
0,SHAP output not empty,True
1,LIME output not empty,True
2,SHAP required columns available,True
3,LIME required columns available,True
4,At least one overlapping model-instance pair,True
5,LIME mapped features are non-missing,True


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/disagreement_input_validation.csv


## 7. Disagreement Configuration

`TOP_K` is the number of top-ranked features compared between SHAP and LIME.

The same `TOP_K` should be used across all datasets for cross-domain consistency.

In [ ]:
# =========================================================
# 7. Disagreement Configuration
# =========================================================

TOP_K = 10

disagreement_config_df = pd.DataFrame([
    {
        "parameter": "RANDOM_STATE",
        "value": RANDOM_STATE,
        "description": "Fixed seed retained for reproducibility. This notebook is deterministic after seeded SHAP and LIME outputs are created."
    },
    {
        "parameter": "TOP_K",
        "value": TOP_K,
        "description": "Number of top SHAP and LIME features compared per model-instance pair."
    },
    {
        "parameter": "agreement_score_formula",
        "value": "(kendall_tau + 1) / 2",
        "description": "Normalises Kendall's Tau from [-1,1] to [0,1] for EES Fidelity."
    }
])

display(disagreement_config_df)
save_df(disagreement_config_df, "disagreement_config.csv")

,parameter,value,description
0,RANDOM_STATE,42,Fixed seed retained for reproducibility. This ...
1,TOP_K,10,Number of top SHAP and LIME features compared ...
2,agreement_score_formula,(kendall_tau + 1) / 2,"Normalises Kendall's Tau from [-1,1] to [0,1] ..."


Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/disagreement_config.csv


PosixPath('/content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/disagreement_config.csv')

## 8. Calculate Instance-Level SHAP-LIME Agreement

For each model and test instance, the notebook compares the top-k SHAP and LIME ranked features.

Outputs include:

- Kendall's Tau
- normalised agreement score
- top-k overlap ratio
- agreement interpretation band

In [ ]:
# =========================================================
# 8. Calculate Instance-Level SHAP-LIME Agreement
# =========================================================

instance_rows = []
comparison_detail_rows = []

shared_keys = sorted(shap_keys.intersection(lime_keys))

for model_id, test_instance_position in shared_keys:

    shap_instance_df = shap_long_df[
        (shap_long_df["model_id"] == model_id) &
        (shap_long_df["test_instance_position"] == test_instance_position)
    ].copy()

    lime_instance_df = lime_long_df[
        (lime_long_df["model_id"] == model_id) &
        (lime_long_df["test_instance_position"] == test_instance_position)
    ].copy()

    if shap_instance_df.empty or lime_instance_df.empty:
        continue

    model_name = shap_instance_df["model_name"].iloc[0] if "model_name" in shap_instance_df.columns else model_id

    metrics = calculate_rank_agreement(
        shap_instance_df=shap_instance_df,
        lime_instance_df=lime_instance_df,
        k=TOP_K
    )

    instance_rows.append({
        "model_id": model_id,
        "model_name": model_name,
        "test_instance_position": test_instance_position,
        **metrics
    })

    shap_top = top_k_rank_table(
        shap_instance_df,
        value_col="abs_shap_value",
        feature_col="feature",
        k=TOP_K
    ).rename(columns={"rank": "shap_rank"})

    lime_top = top_k_rank_table(
        lime_instance_df,
        value_col="abs_lime_weight",
        feature_col="feature",
        k=TOP_K
    ).rename(columns={"rank": "lime_rank"})

    detail_df = shap_top.merge(lime_top, on="feature", how="outer")
    detail_df["model_id"] = model_id
    detail_df["model_name"] = model_name
    detail_df["test_instance_position"] = test_instance_position

    comparison_detail_rows.extend(detail_df.to_dict("records"))

instance_level_agreement_df = pd.DataFrame(instance_rows)
instance_feature_comparison_detail_df = pd.DataFrame(comparison_detail_rows)

display(instance_level_agreement_df.head())
display(instance_feature_comparison_detail_df.head())

print("Instance-level agreement shape:", instance_level_agreement_df.shape)
print("Feature comparison detail shape:", instance_feature_comparison_detail_df.shape)

,model_id,model_name,test_instance_position,kendall_tau,kendall_pvalue,agreement_score,agreement_band,top_k_overlap_count,top_k_overlap_ratio,n_shap_top_features,n_lime_top_features,n_union_features,n_overlap_features,overlap_features
0,decision_tree,Decision Tree,0,-0.178947,0.380708,0.410526,Weak agreement,5,0.500000,10,10,15,5,"age, c_charge_degree_(F2), c_days_from_compas,..."
1,decision_tree,Decision Tree,1,-0.019048,0.923955,0.490476,Weak agreement,4,0.400000,10,10,16,4,"age, c_days_from_compas, days_b_screening_arre..."
2,decision_tree,Decision Tree,2,-0.191304,0.327639,0.404348,Weak agreement,3,0.300000,10,10,17,3,"age, c_days_from_compas, days_b_screening_arrest"
3,decision_tree,Decision Tree,3,-0.139130,0.476519,0.430435,Weak agreement,3,0.300000,10,10,17,3,"age, c_days_from_compas, days_b_screening_arrest"
4,decision_tree,Decision Tree,4,-0.069565,0.721873,0.465217,Weak agreement,3,0.300000,10,10,17,3,"age, days_b_screening_arrest, priors_count"


,feature,abs_shap_value,shap_rank,abs_lime_weight,lime_rank,model_id,model_name,test_instance_position
0,age,0.184688,1.000000,0.132267,4.000000,decision_tree,Decision Tree,0
1,c_charge_degree_(CO3),NaN,NaN,0.144897,2.000000,decision_tree,Decision Tree,0
2,c_charge_degree_(F2),0.000353,7.000000,0.041700,10.000000,decision_tree,Decision Tree,0
3,c_charge_degree_(F5),NaN,NaN,0.141028,3.000000,decision_tree,Decision Tree,0
4,c_charge_degree_(MO3),NaN,NaN,0.064052,7.000000,decision_tree,Decision Tree,0


Instance-level agreement shape: (17500, 14)
Feature comparison detail shape: (283862, 8)


## 9. Model-Level SHAP-LIME Agreement Summary

This summary is the key output for the EES Fidelity pillar.

The model-level `mean_agreement_score` can be used as the SHAP-LIME Agreement component in Fidelity scoring.

In [10]:
# =========================================================
# 9. Model-Level SHAP-LIME Agreement Summary
# =========================================================

if instance_level_agreement_df.empty:
    model_level_agreement_df = pd.DataFrame()
else:
    model_level_agreement_df = (
        instance_level_agreement_df
        .groupby(["model_id", "model_name"], as_index=False)
        .agg(
            n_instances=("test_instance_position", "nunique"),
            mean_kendall_tau=("kendall_tau", "mean"),
            median_kendall_tau=("kendall_tau", "median"),
            min_kendall_tau=("kendall_tau", "min"),
            max_kendall_tau=("kendall_tau", "max"),
            mean_agreement_score=("agreement_score", "mean"),
            median_agreement_score=("agreement_score", "median"),
            min_agreement_score=("agreement_score", "min"),
            max_agreement_score=("agreement_score", "max"),
            mean_top_k_overlap_ratio=("top_k_overlap_ratio", "mean"),
            median_top_k_overlap_ratio=("top_k_overlap_ratio", "median")
        )
    )

    model_level_agreement_df["agreement_score_pct"] = model_level_agreement_df["mean_agreement_score"] * 100
    model_level_agreement_df["agreement_rank"] = (
        model_level_agreement_df["mean_agreement_score"]
        .rank(ascending=False, method="dense")
        .astype(int)
    )
    model_level_agreement_df["agreement_band"] = model_level_agreement_df["mean_agreement_score"].apply(classify_agreement)
    model_level_agreement_df = model_level_agreement_df.sort_values(["agreement_rank", "model_id"]).reset_index(drop=True)

display(model_level_agreement_df)

,model_id,model_name,n_instances,mean_kendall_tau,median_kendall_tau,min_kendall_tau,max_kendall_tau,mean_agreement_score,median_agreement_score,min_agreement_score,max_agreement_score,mean_top_k_overlap_ratio,median_top_k_overlap_ratio,agreement_score_pct,agreement_rank,agreement_band
0,decision_tree,Decision Tree,3500,0.023010,0.038095,-0.469565,0.480000,0.511505,0.519048,0.265217,0.740000,0.448543,0.400000,51.150489,1,Weak agreement
1,random_forest,Random Forest,3500,-0.121791,-0.115789,-0.674074,0.346667,0.439104,0.442105,0.162963,0.673333,0.383429,0.400000,43.910428,2,Weak agreement
2,xgboost,XGBoost,3500,-0.129536,-0.115789,-0.536000,0.247059,0.435232,0.442105,0.232000,0.623529,0.428143,0.400000,43.523177,3,Weak agreement
3,mlp,MLP,3500,-0.309864,-0.313043,-0.689655,0.178947,0.345068,0.343478,0.155172,0.589474,0.288286,0.300000,34.506810,4,Poor agreement
4,logistic_regression,Logistic Regression,3500,-0.326682,-0.323810,-0.696000,0.035294,0.336659,0.338095,0.152000,0.517647,0.341257,0.300000,33.665877,5,Poor agreement


## 10. Fidelity-Ready Agreement Output

This table is specifically designed for `08_02_ees_framework_fidelity_score`.

It exposes a clean model-level score:

```text
shap_lime_agreement_score = mean_agreement_score
```

In [11]:
# =========================================================
# 10. Fidelity-Ready Agreement Output
# =========================================================

if model_level_agreement_df.empty:
    shap_lime_agreement_for_fidelity_df = pd.DataFrame()
else:
    shap_lime_agreement_for_fidelity_df = model_level_agreement_df[
        [
            "model_id",
            "model_name",
            "n_instances",
            "mean_kendall_tau",
            "mean_agreement_score",
            "agreement_score_pct",
            "agreement_rank",
            "agreement_band",
            "mean_top_k_overlap_ratio"
        ]
    ].copy()

    shap_lime_agreement_for_fidelity_df = shap_lime_agreement_for_fidelity_df.rename(
        columns={
            "mean_agreement_score": "shap_lime_agreement_score",
            "agreement_score_pct": "shap_lime_agreement_score_pct",
            "agreement_rank": "shap_lime_agreement_rank",
            "agreement_band": "shap_lime_agreement_band"
        }
    )

display(shap_lime_agreement_for_fidelity_df)

,model_id,model_name,n_instances,mean_kendall_tau,shap_lime_agreement_score,shap_lime_agreement_score_pct,shap_lime_agreement_rank,shap_lime_agreement_band,mean_top_k_overlap_ratio
0,decision_tree,Decision Tree,3500,0.023010,0.511505,51.150489,1,Weak agreement,0.448543
1,random_forest,Random Forest,3500,-0.121791,0.439104,43.910428,2,Weak agreement,0.383429
2,xgboost,XGBoost,3500,-0.129536,0.435232,43.523177,3,Weak agreement,0.428143
3,mlp,MLP,3500,-0.309864,0.345068,34.506810,4,Poor agreement,0.288286
4,logistic_regression,Logistic Regression,3500,-0.326682,0.336659,33.665877,5,Poor agreement,0.341257


In [12]:
# =========================================================
# 11. Interpretation Table
# =========================================================

agreement_interpretation_table_df = pd.DataFrame([
    {
        "agreement_score_range": "0.85 to 1.00",
        "band": "Very strong agreement",
        "interpretation": "SHAP and LIME provide highly consistent feature-rank explanations."
    },
    {
        "agreement_score_range": "0.70 to <0.85",
        "band": "Strong agreement",
        "interpretation": "SHAP and LIME mostly identify similar explanatory drivers."
    },
    {
        "agreement_score_range": "0.55 to <0.70",
        "band": "Moderate agreement",
        "interpretation": "SHAP and LIME partly agree, but explanation differences remain."
    },
    {
        "agreement_score_range": "0.40 to <0.55",
        "band": "Weak agreement",
        "interpretation": "SHAP and LIME provide noticeably different explanation rankings."
    },
    {
        "agreement_score_range": "0.00 to <0.40",
        "band": "Poor agreement",
        "interpretation": "SHAP and LIME provide inconsistent or contradictory explanations."
    }
])

display(agreement_interpretation_table_df)

,agreement_score_range,band,interpretation
0,0.85 to 1.00,Very strong agreement,SHAP and LIME provide highly consistent featur...
1,0.70 to <0.85,Strong agreement,SHAP and LIME mostly identify similar explanat...
2,0.55 to <0.70,Moderate agreement,"SHAP and LIME partly agree, but explanation di..."
3,0.40 to <0.55,Weak agreement,SHAP and LIME provide noticeably different exp...
4,0.00 to <0.40,Poor agreement,SHAP and LIME provide inconsistent or contradi...


## 12. Save Outputs

In [13]:
# =========================================================
# 12. Save Outputs
# =========================================================

save_df(instance_level_agreement_df, "instance_level_shap_lime_agreement.csv")
save_df(model_level_agreement_df, "model_level_shap_lime_agreement_summary.csv")
save_df(shap_lime_agreement_for_fidelity_df, "shap_lime_agreement_for_fidelity.csv")
save_df(instance_feature_comparison_detail_df, "instance_feature_comparison_detail.csv")
save_df(agreement_interpretation_table_df, "agreement_interpretation_table.csv")

save_df(instance_level_agreement_df, "instance_level_shap_lime_agreement.csv", folder=DATASETS_PATH)
save_df(model_level_agreement_df, "model_level_shap_lime_agreement_summary.csv", folder=DATASETS_PATH)
save_df(shap_lime_agreement_for_fidelity_df, "shap_lime_agreement_for_fidelity.csv", folder=DATASETS_PATH)
save_df(instance_feature_comparison_detail_df, "instance_feature_comparison_detail.csv", folder=DATASETS_PATH)
save_df(agreement_interpretation_table_df, "agreement_interpretation_table.csv", folder=DATASETS_PATH)

# Compatibility names for older downstream notebooks
save_df(model_level_agreement_df, "main_top10_disagreement_summary.csv")
save_df(model_level_agreement_df, "main_top10_disagreement_summary.csv", folder=DATASETS_PATH)
save_df(model_level_agreement_df, "model_level_disagreement_summary.csv")
save_df(model_level_agreement_df, "model_level_disagreement_summary.csv", folder=DATASETS_PATH)

save_excel(
    {
        "agreement_for_fidelity": shap_lime_agreement_for_fidelity_df,
        "model_level_agreement": model_level_agreement_df,
        "instance_level_agreement": instance_level_agreement_df,
        "feature_comparison_detail": instance_feature_comparison_detail_df,
        "feature_standardisation": feature_standardisation_audit_df,
        "interpretation_table": agreement_interpretation_table_df,
        "input_validation": disagreement_input_validation_df,
        "config": disagreement_config_df
    },
    f"shap_lime_disagreement_{DATASET_NAME}_outputs.xlsx"
)

manifest = {
    "dataset_name": DATASET_NAME,
    "dataset_label": CONFIG["dataset_label"],
    "notebook_name": NOTEBOOK_NAME,
    "run_timestamp": datetime.now().isoformat(),
    "random_state": RANDOM_STATE,
    "top_k": TOP_K,
    "agreement_score_formula": "(kendall_tau + 1) / 2",
    "fidelity_use": (
        "The output shap_lime_agreement_for_fidelity.csv provides the SHAP-LIME Agreement component "
        "for the revised EES Fidelity pillar."
    ),
    "standardisation_note": (
        "This notebook uses the same configurable pipeline and rank-agreement logic across German Credit, "
        "Pima Diabetes and COMPAS."
    )
}

with open(LOGS_PATH / "shap_lime_disagreement_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

with open(TABLES_PATH / "shap_lime_disagreement_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=4)

print("All SHAP-LIME disagreement outputs saved.")

Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/instance_level_shap_lime_agreement.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/model_level_shap_lime_agreement_summary.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/shap_lime_agreement_for_fidelity.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/instance_feature_comparison_detail.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables/agreement_interpretation_table.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/datasets/instance_level_shap_lime_agreement.csv
Saved: /content/drive/MyDrive/Master_Research/07_Results/compas

## 13. Interpretation Note

This notebook does not generate new explanations. It evaluates whether two independent explanation methods tell a similar story.

The key research contribution is that SHAP-LIME agreement is converted into a model-level quantitative score that can be incorporated into the EES Fidelity pillar.

This strengthens the Fidelity pillar by adding cross-method explanation consistency alongside SHAP reconstruction fidelity and LIME local surrogate fidelity.

In [14]:
# =========================================================
# 13. Final Completion Message
# =========================================================

print("=" * 100)
print(f"06 SHAP-LIME Disagreement completed successfully for: {CONFIG['dataset_label']}")
print("=" * 100)

print(f"Tables saved to:   {TABLES_PATH}")
print(f"Datasets saved to: {DATASETS_PATH}")
print(f"Logs saved to:     {LOGS_PATH}")

print("Key outputs:")
print("- shap_lime_agreement_for_fidelity.csv")
print("- model_level_shap_lime_agreement_summary.csv")
print("- instance_level_shap_lime_agreement.csv")
print("- instance_feature_comparison_detail.csv")
print("- main_top10_disagreement_summary.csv")
print(f"- shap_lime_disagreement_{DATASET_NAME}_outputs.xlsx")

06 SHAP-LIME Disagreement completed successfully for: COMPAS
Tables saved to:   /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/tables
Datasets saved to: /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/datasets
Logs saved to:     /content/drive/MyDrive/Master_Research/07_Results/compas/06_shap_lime_explaination_disagreement_compas/logs
Key outputs:
- shap_lime_agreement_for_fidelity.csv
- model_level_shap_lime_agreement_summary.csv
- instance_level_shap_lime_agreement.csv
- instance_feature_comparison_detail.csv
- main_top10_disagreement_summary.csv
- shap_lime_disagreement_compas_outputs.xlsx
